## Setup

In [1]:
pip install databricks-sql-connector


  Using cached oauthlib-3.3.1-py3-none-any.whl.metadata (7.9 kB)
  Using cached openpyxl-3.1.5-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
  Using cached et_xmlfile-2.0.0-py3-none-any.whl.metadata (2.7 kB)
  Using cached charset_normalizer-3.5.1-cp312-cp312-macosx_10_13_universal2.whl.metadata (45 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached certifi-2026.7.22-py3-none-any.whl.metadata (2.5 kB)
Using cached oauthlib-3.3.1-py3-none-any.whl (160 kB)
Using cached openpyxl-3.1.5-py2.py3-none-any.whl (250 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 36.6 MB/s eta 0:00:00MB/s eta 0:00:01
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)
Using cached certifi-2026.7.22-py3-none-any.whl (136 kB)
Using cached charset_normalizer-3.5.1-cp312-cp312-macosx_10_13_universal2

In [8]:
pip install --upgrade "databricks-connect==14.3.*"


  Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl.metadata (61 kB)
  Using cached pandas-2.3.3-cp312-cp312-macosx_10_13_x86_64.whl.metadata (91 kB)
  Using cached protobuf-6.33.6-cp39-abi3-macosx_10_9_universal2.whl.metadata (593 bytes)
  Using cached pyasn1_modules-0.4.2-py3-none-any.whl.metadata (3.5 kB)
  Using cached cryptography-50.0.1-cp312-abi3-macosx_10_12_x86_64.whl
  Using cached cffi-2.1.1-cp312-cp312-macosx_10_15_x86_64.whl.metadata (2.5 kB)
  Using cached pyasn1-0.6.4-py3-none-any.whl.metadata (8.4 kB)
  Using cached pycparser-3.0-py3-none-any.whl.metadata (8.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 6.9 MB/s eta 0:00:008 MB/s eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 35.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 43.8 MB/s eta 0:00:000:00:01
Using cached numpy-1.26.4-cp312-cp312-macosx_10_9_x86_64.whl (20.3 MB)
Using cached pandas-2.3.3-cp312-cp312-macosx_10_13_x86_64.whl (11.6 MB)
  

In [9]:
pip uninstall pyspark

Note: you may need to restart the kernel to use updated packages.


In [11]:
pip install dotenv

  Using cached dotenv-0.9.9-py2.py3-none-any.whl.metadata (279 bytes)
  Using cached python_dotenv-1.2.3-py3-none-any.whl.metadata (29 kB)
Using cached dotenv-0.9.9-py2.py3-none-any.whl (1.9 kB)
Using cached python_dotenv-1.2.3-py3-none-any.whl (22 kB)

[notice] A new release of pip is available: 24.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Libraries

In [10]:
from databricks import sql
import os
from dotenv import load_dotenv
import pandas as pd
from pathlib import Path
import json
import urllib.parse
import unicodedata

import re

load_dotenv()

True

# Databricks connection

In [ ]:


SERVER_HOSTNAME = os.getenv("SERVER_HOSTNAME")
DATABRICKS_TOKEN = os.getenv("DATABRICKS_TOKEN")
SERVER_HTTP_PATH = os.getenv("SERVER_HTTP_PATH")

if not DATABRICKS_TOKEN or not SERVER_HTTP_PATH or not SERVER_HOSTNAME:
    raise ValueError("DATABRICKS_TOKEN and SERVER_HTTP_PATH and SERVER_HOSTNAME must be set in the .env file")

In [23]:
from databricks import sql

connection = sql.connect(
    server_hostname=SERVER_HOSTNAME,
    http_path=SERVER_HTTP_PATH,
    access_token=DATABRICKS_TOKEN
)

with connection.cursor() as cursor:
    cursor.execute("""
        SELECT *
        FROM reg_bronze_dev.data_patagonia.camarastrampa_idx
    """)

    df = cursor.fetchall_arrow().to_pandas()

connection.close()
df.head()

# save
df.to_csv("camarastrampa_idx.csv", index=False)

In [ ]:
df.source_web_url.iloc[0].strip()

'https://uccl0.sharepoint.com/sites/sh365_epiiUC/Documentos%20compartidos/01_EPII%20UC/07_DatosRevisi%C3%B3n/05_Camarastrampa/C%C3%A1maras%20Trampa%20Snapshot%202025-26/CAM04/101EK113/02210873.JPG'

## Data analysis - Drive and Databricks

### Load dataframes from existing files

In [2]:

# Databricks dataframe
DB_CAMERAS_PATH = Path("data/camarastrampa_idx.csv")
db_df = pd.read_csv(DB_CAMERAS_PATH)

# Google Drive dataframe
SP_CAMERAS_PATH = Path("data/wild_detector_results.json")
with open(SP_CAMERAS_PATH, "r") as f:
    sp_data = json.load(f)

    # Preprocess this information, extracting the source path from the values
    sp_source_list = []
    for dfolder in sp_data:
        for dfile in sp_data[dfolder]:
            sp_source_list.append(dfile["source_path"].strip())

    # Create a DataFrame from the extracted source paths
    sp_df = pd.DataFrame(sp_source_list, columns=["source_web_url"])

The origin folder of the Google drive images is 'sin-clasificar', and it was retrieved directly from the 
Snapshot folder. The this iamges where classified using bboxes and the results were stored in the Training dataset. 

The source url (sharepoint) of the images is in the wild_detector_results.json. 

### Create sharepoint file df

In [68]:
db_cols = {
    "cameras": [],
    "subfolder": [],
    "file": [],
    'filename': []
}

for line in db_df.itertuples():
    source_web_url = urllib.parse.unquote(line.source_web_url).split("/")
    camera, dcim, subfolder, filename = (
        source_web_url[-4],
        source_web_url[-3],
        source_web_url[-2],
        source_web_url[-1],
    )
    # 1. if dcim starts with CAM, then dcim is none
    if dcim.startswith('CAM'):
        camera = dcim
        dcim = None
    # 2. if the subfolder starts with CAM and the len of filename stripped with 3 > 1
    if subfolder.startswith('CAM') and len(filename.split("_")) > 1:
        camera = subfolder
        subfolder = None
        dcim = None
        filename = filename.split("_")[-1]
    # 3. if the file has _ between then normalize it (the subfolder doesn't start with CAM)
    if len(filename.split("_")) > 1:
        filename = filename.split("_")[-1]
        
    if dcim == "DCIM":
        dcim = None
    # if dcim != None and dcim != 'DCIM':
    if subfolder and subfolder.startswith('CAMARA TORRE'):
        camera = 'CAMARA_TORRE'
        subfolder, dcim = None, None
    
    if re.match(r"^CAM\s*\d{1,2}$", camera) and camera != 'CAMARA_TORRE':
        camera = camera.replace(" ", "").upper()
        number = int(camera.replace("CAM", ""))
        camera = f"CAM{number:02d}"

    if subfolder and  len(subfolder.split(" ")) > 1:
        subfolder = subfolder.split(" ")[0]
        

    db_cols['cameras'].append(camera)
    db_cols['subfolder'].append(subfolder)
    db_cols['file'].append(filename.split('.')[0])
    db_cols['filename'].append(filename)

db_df_new = pd.DataFrame(db_cols)

In [69]:
db_df_new.head()

,cameras,subfolder,file,filename
0,CAM04,101EK113,02210873,02210873.JPG
1,CAM04,101EK113,01070218,01070218.JPG
2,CAM04,100EK113,01040745,01040745.JPG
3,CAM04,100EK113,12090333,12090333.JPG
4,CAM04,102EK113,02230772,02230772.JPG


In [70]:
db_df_new.subfolder.unique()

array(['101EK113', '100EK113', '102EK113', '103EK113', '104EK113', None],
      dtype=object)

In [37]:
sorted(db_df_new.cameras.unique().tolist())

['CAM01',
 'CAM02',
 'CAM03',
 'CAM04',
 'CAM05',
 'CAM06',
 'CAM07',
 'CAM08',
 'CAM09',
 'CAM10',
 'CAM11',
 'CAM12',
 'CAM13',
 'CAM14',
 'CAM15',
 'CAM16',
 'CAMARA_TORRE']

### Create databricks file df

In [71]:
sp_cols = {
    'cameras': [],
    'subfolder': [],
    'file': [],
    'filename': [],
}
for line in sp_df.itertuples():
    source_web_url = line.source_web_url.split("/")
    filename = source_web_url[-1]
    subfolder = source_web_url[-2]
    dcim = source_web_url[-3]
    camera = source_web_url[-4]

    # Without dcim and subfolder
    if subfolder.startswith("CAM"):
        camera = subfolder
        dcim, subfolder = None, None
    # Dcim doesn't exist
    elif dcim.startswith("CAM"):
        camera = dcim
        dcim = None
    # Exists dcim
    else:
        pass

    if camera == "CAM01_02":
        camera = "CAM01,CAM02"
    
    sp_cols['cameras'].append(camera)
    sp_cols['subfolder'].append(subfolder)
    sp_cols['file'].append(filename)
    sp_cols['filename'].append(filename.split('.')[0])
    
gd_df_new = pd.DataFrame(sp_cols)

In [72]:
sorted(gd_df_new.cameras.unique().tolist())

['CAM01,CAM02',
 'CAM03',
 'CAM04',
 'CAM05',
 'CAM07',
 'CAM08',
 'CAM09',
 'CAM10',
 'CAM11',
 'CAM12',
 'CAM13',
 'CAM14',
 'CAM15',
 'CAM16']

## Compare the two dataframes

In [73]:
# Compare folders that exists in databricks but not in drive
def prepare_comparison(df, filename_column):
    result = df.copy()

    # Separar casos como CAM01,CAM02 en dos registros
    result["camera"] = (
        result["cameras"]
        .fillna("")
        .astype(str)
        .str.split(",")
    )

    result = result.explode("camera")

    result["camera"] = result["camera"].str.strip().str.upper()

    # Normalizar el nombre sin extensión
    result["file_key"] = (
        result[filename_column]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.rsplit(".", n=1)
        .str[0]
        .str.upper()
    )

    return result[["camera", "file_key"]].drop_duplicates()



def prepare_comparison_2(df, filename_column):
    # Considering subfolder
    result = df.copy()

    # Separar casos como CAM01,CAM02
    result["camera"] = (
        result["cameras"]
        .fillna("")
        .astype(str)
        .str.split(",")
    )

    result = result.explode("camera")

    result["camera"] = (
        result["camera"]
        .str.strip()
        .str.upper()
    )

    # Normalizar subfolder
    result["subfolder_key"] = (
        result["subfolder"]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.upper()
    )

    # Normalizar nombre del archivo
    result["file_key"] = (
        result[filename_column]
        .fillna("")
        .astype(str)
        .str.strip()
        .str.rsplit(".", n=1)
        .str[0]
        .str.upper()
    )

    return result[
        ["camera", "subfolder_key", "file_key"]
    ].drop_duplicates()


In [83]:
# Archivos de Databricks 
db_compare = prepare_comparison_2(
    db_df_new,
    filename_column="file",
)

# Archivos presentes en entrenamiento (GDrive)
gd_compare = prepare_comparison_2(
    gd_df_new,
    filename_column="file",
)

# Check cameras which are in databricks but not in gdrive
db_cameras = set(db_compare["camera"])
gd_cameras = set(gd_compare["camera"])

cameras_without_files = sorted(db_cameras - gd_cameras)

print(cameras_without_files)


['CAM06', 'CAMARA_TORRE']


In [79]:
special_cameras = {"CAM01", "CAM02"}

# CAM01 y CAM02: comparar sin subfolder
db_special = (
    db_compare[
        db_compare["camera"].isin(special_cameras)
    ][["camera", "file_key"]]
    .drop_duplicates()
)

gd_special = (
    gd_compare[
        gd_compare["camera"].isin(special_cameras)
    ][["camera", "file_key"]]
    .drop_duplicates()
)

missing_special = (
    db_special
    .merge(
        gd_special,
        on=["camera", "file_key"],
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
    .assign(subfolder_key="")
)

# Otras cámaras: comparar camera + subfolder + file
db_regular = db_compare[
    ~db_compare["camera"].isin(special_cameras)
]

gd_regular = gd_compare[
    ~gd_compare["camera"].isin(special_cameras)
]

missing_regular = (
    db_regular
    .merge(
        gd_regular,
        on=["camera", "subfolder_key", "file_key"],
        how="left",
        indicator=True,
    )
    .query("_merge == 'left_only'")
    .drop(columns="_merge")
)

# Resultado final
missing_files = pd.concat(
    [
        missing_special[
            ["camera", "subfolder_key", "file_key"]
        ],
        missing_regular[
            ["camera", "subfolder_key", "file_key"]
        ],
    ],
    ignore_index=True,
).sort_values(
    ["camera", "subfolder_key", "file_key"]
)

# Totales comparables
total_databricks = (
    len(db_special) + len(db_regular)
)

total_missing = len(missing_files)
total_in_gdrive = total_databricks - total_missing

print(f"Total de archivos en Databricks: {total_databricks}")
print(f"Archivos encontrados en GDrive: {total_in_gdrive}")
print(f"Archivos faltantes en GDrive: {total_missing}")
print(
    f"Porcentaje faltante: "
    f"{total_missing / total_databricks * 100:.2f}%"
)

missing_files

Total de archivos en Databricks: 10724
Archivos encontrados en GDrive: 6592
Archivos faltantes en GDrive: 4132
Porcentaje faltante: 38.53%


,camera,subfolder_key,file_key
9,CAM01,,01072022
3,CAM01,,03062022
8,CAM01,,04062022
1,CAM01,,05052022
4,CAM01,,12042022
...,...,...,...
4125,CAMARA_TORRE,,06050297
4128,CAMARA_TORRE,,06050298
4130,CAMARA_TORRE,,06050299
4127,CAMARA_TORRE,,07110398


In [81]:
missing_files.to_csv('data/missing_files_total.csv', index=False)

In [82]:
files_by_camera_missing = (
    missing_files
    .groupby("camera")
    .size()
    .reset_index(name="total_files")
    .sort_values("camera")
)

files_by_camera_missing

,camera,total_files
0,CAM01,12
1,CAM02,22
2,CAM03,593
3,CAM04,3401
4,CAM05,20
5,CAM06,16
6,CAM08,60
7,CAM09,1
8,CAMARA_TORRE,7


In [88]:
string_files = files_by_camera_missing.to_string()
with open('data/summary.txt', 'a') as file:
    file.write(string_files)